In [10]:
import pandas as pd
from collections import Counter
from itertools import combinations
import sys
import os

In [ ]:
path = 'Raw_Data'

orders = pd.read_csv(os.path.join(path, 'orders.csv'))


op_prior = pd.read_csv(os.path.join(path, 'order_products__prior.csv'), usecols=['order_id', 'product_id'])
products = pd.read_csv(os.path.join(path, 'products.csv'), usecols=['product_id', 'product_name'])


In [12]:
product_name_map = products.set_index('product_id')['product_name'].to_dict()

# Grouping 
print("Grouping transactions...")
transactions = op_prior.groupby('order_id')['product_id'].apply(list)

# Counting Pairs 
print("Counting pairs & products...")
pair_counts = Counter()
product_counts = Counter()

for i, products_in_order in enumerate(transactions):
    
    # Support A, Support 
    product_counts.update(products_in_order)
    products_in_order.sort()
    
    if len(products_in_order) >= 2:
        pair_counts.update(combinations(products_in_order, 2))
        
    if (i + 1) % 100000 == 0:
        print(f"Processed {i + 1} orders...")

# Create DataFrame 
print("Creating DataFrame...")
df_pairs = pd.DataFrame.from_dict(pair_counts, orient='index', columns=['frequency']).reset_index()

df_pairs[['product_id_A', 'product_id_B']] = pd.DataFrame(df_pairs['index'].tolist(), index=df_pairs.index)
df_pairs = df_pairs.drop(columns=['index'])

# Filtering 
print("Filtering low frequency...")
df_pairs = df_pairs[df_pairs['frequency'] >= 50].copy()

# Mapping & Metrics 
print("Calculating Metrics...")

product_counts_dict = dict(product_counts)
df_pairs['orders_A'] = df_pairs['product_id_A'].map(product_counts_dict)
df_pairs['orders_B'] = df_pairs['product_id_B'].map(product_counts_dict)

#
df_pairs['product_name_A'] = df_pairs['product_id_A'].map(product_name_map)
df_pairs['product_name_B'] = df_pairs['product_id_B'].map(product_name_map)

#
n_total_transactions = len(transactions)

# Confidence (A -> B) = P(A & B) / P(A)
df_pairs['confidence'] = df_pairs['frequency'] / df_pairs['orders_A']

# Lift = Confidence / P(B)
prob_B = df_pairs['orders_B'] / n_total_transactions
df_pairs['lift'] = df_pairs['confidence'] / prob_B

Grouping transactions...
Counting pairs & products...
Processed 100000 orders...
Processed 200000 orders...
Processed 300000 orders...
Processed 400000 orders...
Processed 500000 orders...
Processed 600000 orders...
Processed 700000 orders...
Processed 800000 orders...
Processed 900000 orders...
Processed 1000000 orders...
Processed 1100000 orders...
Processed 1200000 orders...
Processed 1300000 orders...
Processed 1400000 orders...
Processed 1500000 orders...
Processed 1600000 orders...
Processed 1700000 orders...
Processed 1800000 orders...
Processed 1900000 orders...
Processed 2000000 orders...
Processed 2100000 orders...
Processed 2200000 orders...
Processed 2300000 orders...
Processed 2400000 orders...
Processed 2500000 orders...
Processed 2600000 orders...
Processed 2700000 orders...
Processed 2800000 orders...
Processed 2900000 orders...
Processed 3000000 orders...
Processed 3100000 orders...
Processed 3200000 orders...
Creating DataFrame...
Filtering low frequency...
Calculatin

In [ ]:
process_path = 'Process_Data'
df_pairs.to_csv(os.path.join(process_path, 'market_basket.csv'),index=False)
